In [ ]:
!pip install -q "streamlit>=1.31" transformers torchaudio

# cloudflared = Cloudflare's tunnel client. Reliable proxying for Streamlit assets.
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

cloudflared version 2026.3.0 (built 2026-03-09-14:08 UTC)


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU. Predictions will be slower but still work.")

CUDA available: True
GPU: Tesla T4


**Alternative — load from Google Drive:**

In [ ]:
# Uncomment if pulling from Google Drive instead:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
os.makedirs("checkpoints", exist_ok=True)
shutil.copy(
    "/content/drive/MyDrive/ADI/checkpoints/finetune_best.pt",
    "checkpoints/finetune_best.pt",
)
print("Checkpoint loaded from Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoint loaded from Drive.


In [ ]:
%%writefile app.py
"""Voice Emotion Sense — calm, earthy speech emotion recognition."""

import os
import io
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import streamlit as st
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor

# ============================================================
# Configuration
# ============================================================
PRETRAINED_MODEL  = "facebook/wav2vec2-base"
HIDDEN_DIM        = 768
NUM_CLASSES       = 8
SAMPLE_RATE       = 16000
MAX_AUDIO_SECONDS = 4.0
MAX_AUDIO_SAMPLES = int(SAMPLE_RATE * MAX_AUDIO_SECONDS)
DEVICE            = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT_PATH   = "checkpoints/finetune_best.pt"

EMOTION_MAP = {
    "01": "neutral",  "02": "calm",     "03": "happy",   "04": "sad",
    "05": "angry",    "06": "fearful",  "07": "disgust", "08": "surprised",
}
EMOTION_TO_IDX = {name: idx for idx, name in enumerate(sorted(EMOTION_MAP.values()))}
IDX_TO_EMOTION = {idx: name for name, idx in EMOTION_TO_IDX.items()}

EMOTION_DISPLAY = {
    "neutral":   {"emoji": "😐", "color": "#A8B5A0", "description": "calm, even-keeled"},
    "calm":      {"emoji": "🌿", "color": "#87A878", "description": "peaceful, settled"},
    "happy":     {"emoji": "🌻", "color": "#E0A458", "description": "joyful, warm"},
    "sad":       {"emoji": "🌧️", "color": "#7A8B99", "description": "low, melancholy"},
    "angry":     {"emoji": "🔥", "color": "#C5705D", "description": "intense, heated"},
    "fearful":   {"emoji": "🌫️", "color": "#9B8AA0", "description": "anxious, uneasy"},
    "disgust":   {"emoji": "🍂", "color": "#9C7A4B", "description": "averse, repulsed"},
    "surprised": {"emoji": "✨", "color": "#D4A574", "description": "startled, alert"},
}

# ============================================================
# Page setup + custom CSS theme
# ============================================================
st.set_page_config(
    page_title="VibeCheck",
    page_icon="🍃",
    layout="centered",
)

EARTHY_CSS = '''
<style>
    .stApp {
        background: linear-gradient(180deg, #FBF7F0 0%, #F4EBD9 100%);
    }
    #MainMenu {visibility: hidden;}
    footer {visibility: hidden;}
    header {visibility: hidden;}
    html, body, [class*="css"] {
        font-family: "Georgia", "Times New Roman", serif;
        color: #4A453E;
    }
    h1, h2, h3 {
        font-family: "Georgia", serif;
        color: #4A453E;
        font-weight: 400;
        letter-spacing: 0.5px;
    }
    .stTabs [data-baseweb="tab-list"] {
        gap: 8px;
        background: transparent;
        border-bottom: 1px solid rgba(135, 168, 120, 0.25);
    }
    .stTabs [data-baseweb="tab"] {
        background-color: transparent;
        color: #7A6F62;
        font-family: "Georgia", serif;
        font-size: 16px;
        padding: 12px 24px;
        border-radius: 0;
        border-bottom: 2px solid transparent;
    }
    .stTabs [aria-selected="true"] {
        color: #5A7A56 !important;
        border-bottom-color: #5A7A56 !important;
        font-weight: 600;
        background-color: transparent !important;
    }
    .stButton > button, .stDownloadButton > button {
        background: linear-gradient(135deg, #87A878 0%, #5A7A56 100%);
        color: #FBF7F0;
        border: none;
        border-radius: 28px;
        padding: 10px 28px;
        font-family: "Georgia", serif;
        font-size: 14px;
        letter-spacing: 0.6px;
        transition: all 0.2s ease;
    }
    .stButton > button:hover {
        transform: translateY(-1px);
        box-shadow: 0 4px 12px rgba(90, 122, 86, 0.3);
    }
    [data-testid="stFileUploader"] section,
    [data-testid="stAudioInput"] {
        background-color: rgba(240, 230, 213, 0.4);
        border: 1.5px dashed rgba(135, 168, 120, 0.4);
        border-radius: 14px;
    }
    audio { width: 100%; margin-top: 12px; }
    [data-baseweb="notification"] {
        background-color: rgba(255, 253, 248, 0.7) !important;
        border-left: 4px solid #87A878 !important;
        border-radius: 8px !important;
    }
    .stProgress > div > div > div { background-color: #87A878; }
</style>
'''
st.markdown(EARTHY_CSS, unsafe_allow_html=True)

# ============================================================
# Header
# ============================================================
HEADER = '''
<div style="text-align: center; padding: 24px 0 16px 0;">
    <h1 style="font-size: 42px; margin: 0 0 6px 0; letter-spacing: 1.5px;">
        🍃 VibeCheck
    </h1>
    <p style="color: #7A6F62; font-style: italic; font-size: 15px; margin: 0;">
        Listen to the feeling beneath the words
    </p>
</div>
'''
st.markdown(HEADER, unsafe_allow_html=True)

# ============================================================
# Model architecture (must match training)
# ============================================================
class EmotionClassifierHead(nn.Module):
    def __init__(self, input_dim=HIDDEN_DIM, hidden_dim=256,
                 num_classes=NUM_CLASSES, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )
    def forward(self, x):
        return self.net(x)


def masked_mean_pool(hidden_states, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    summed = (hidden_states * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-6)
    return summed / counts


class Wav2Vec2EmotionModel(nn.Module):
    def __init__(self, pretrained_model_name=PRETRAINED_MODEL, num_classes=NUM_CLASSES):
        super().__init__()
        self.encoder = Wav2Vec2Model.from_pretrained(pretrained_model_name)
        self.encoder.feature_extractor._freeze_parameters()
        self.head = EmotionClassifierHead(
            input_dim=self.encoder.config.hidden_size,
            num_classes=num_classes,
        )
    def forward(self, input_values, attention_mask=None):
        outputs = self.encoder(input_values=input_values, attention_mask=attention_mask)
        if attention_mask is not None:
            input_lengths = attention_mask.sum(dim=1)
            output_lengths = self.encoder._get_feat_extract_output_lengths(
                input_lengths
            ).to(torch.long)
            B, T_out, _ = outputs.last_hidden_state.shape
            output_mask = torch.zeros(
                (B, T_out), dtype=torch.long,
                device=outputs.last_hidden_state.device,
            )
            for i, length in enumerate(output_lengths):
                output_mask[i, :length] = 1
            pooled = masked_mean_pool(outputs.last_hidden_state, output_mask)
        else:
            pooled = outputs.last_hidden_state.mean(dim=1)
        return self.head(pooled)


# ============================================================
# Load the model (cached)
# ============================================================
@st.cache_resource
def load_model():
    if not os.path.exists(CHECKPOINT_PATH):
        st.error(f"No checkpoint at {CHECKPOINT_PATH}. Did you upload it?")
        st.stop()
    model = Wav2Vec2EmotionModel().to(DEVICE)
    model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
    model.eval()
    extractor = Wav2Vec2FeatureExtractor.from_pretrained(PRETRAINED_MODEL)
    return model, extractor


with st.spinner("Loading the model..."):
    model, feature_extractor = load_model()


# ============================================================
# Audio preprocessing & prediction
# ============================================================
def preprocess_waveform(waveform, sr):
    if waveform.dim() == 1:
        waveform = waveform.unsqueeze(0)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    if sr != SAMPLE_RATE:
        waveform = torchaudio.transforms.Resample(sr, SAMPLE_RATE)(waveform)
    waveform = waveform.squeeze(0)
    if waveform.size(0) < MAX_AUDIO_SAMPLES:
        waveform = torch.nn.functional.pad(
            waveform, (0, MAX_AUDIO_SAMPLES - waveform.size(0))
        )
    else:
        waveform = waveform[:MAX_AUDIO_SAMPLES]
    return waveform


def load_audio_from_bytes(audio_bytes):
    waveform, sr = torchaudio.load(io.BytesIO(audio_bytes))
    return waveform, sr


def predict(waveform, sr):
    waveform = preprocess_waveform(waveform, sr)
    with torch.no_grad():
        inputs = feature_extractor(
            waveform.numpy(),
            sampling_rate=SAMPLE_RATE,
            return_tensors="pt",
            return_attention_mask=True,
        )
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        logits = model(**inputs)
        probs = F.softmax(logits, dim=-1).squeeze().cpu().numpy()
    return probs


# ============================================================
# Result rendering
# ============================================================
def render_result(probs):
    predicted_idx = int(probs.argmax())
    predicted_emotion = IDX_TO_EMOTION[predicted_idx]
    confidence = float(probs[predicted_idx])
    display = EMOTION_DISPLAY[predicted_emotion]

    color = display["color"]
    emoji = display["emoji"]
    desc = display["description"]
    name_upper = predicted_emotion.upper()
    pct = f"{confidence*100:.1f}"

    headline = f'''
    <div style="
        background: linear-gradient(135deg, #FBF7F0 0%, #F0E6D5 100%);
        border-left: 6px solid {color};
        padding: 28px 32px;
        border-radius: 14px;
        text-align: center;
        margin: 24px 0 20px 0;
        box-shadow: 0 2px 12px rgba(74, 69, 62, 0.06);
    ">
        <div style="font-size: 72px; margin-bottom: 8px;">{emoji}</div>
        <div style="font-size: 32px; color: #4A453E; font-weight: 600; letter-spacing: 1px;">
            {name_upper}
        </div>
        <div style="font-size: 14px; color: #7A6F62; margin-top: 6px; font-style: italic;">
            {desc}
        </div>
        <div style="font-size: 17px; color: #6B5F52; margin-top: 16px;">
            <span style="font-weight: 600;">{pct}%</span> confidence
        </div>
    </div>
    '''
    st.markdown(headline, unsafe_allow_html=True)

    st.markdown(
        '<h3 style="color: #5A7A56; font-size: 17px; margin-top: 24px;">All emotions</h3>',
        unsafe_allow_html=True,
    )

    sorted_indices = probs.argsort()[::-1]
    for idx in sorted_indices:
        emotion = IDX_TO_EMOTION[idx]
        prob = float(probs[idx])
        d = EMOTION_DISPLAY[emotion]
        bar_width = max(prob * 100, 2)
        is_top = (idx == predicted_idx)
        opacity = 1.0 if is_top else 0.7
        bar_emoji = d["emoji"]
        bar_color = d["color"]
        bar_pct = f"{prob*100:.1f}"

        bar_html = f'''
        <div style="margin-bottom: 10px;">
            <div style="display: flex; justify-content: space-between; margin-bottom: 4px;">
                <span style="color: #4A453E; font-size: 14px;">
                    {bar_emoji} <strong>{emotion}</strong>
                </span>
                <span style="color: #7A6F62; font-size: 13px;">{bar_pct}%</span>
            </div>
            <div style="background-color: rgba(240, 230, 213, 0.5); border-radius: 8px; height: 10px; overflow: hidden;">
                <div style="background: {bar_color}; width: {bar_width}%; height: 100%; border-radius: 8px; opacity: {opacity};"></div>
            </div>
        </div>
        '''
        st.markdown(bar_html, unsafe_allow_html=True)


# ============================================================
# Tabs
# ============================================================
tab_record, tab_upload = st.tabs(["🎙️  Record", "📁  Upload"])

with tab_record:
    st.markdown(
        '<h3 style="color: #5A7A56; font-size: 18px; margin-top: 8px;">Record your voice</h3>',
        unsafe_allow_html=True,
    )
    st.markdown(
        '<p style="color: #7A6F62; font-style: italic; font-size: 14px; margin-bottom: 16px;">'
        'Tap the microphone to start recording. Tap again to stop.</p>',
        unsafe_allow_html=True,
    )

    audio_value = st.audio_input("Tap to record")

    if audio_value is not None:
        audio_bytes = audio_value.read()
        with st.spinner("Listening..."):
            waveform, sr = load_audio_from_bytes(audio_bytes)
            probs = predict(waveform, sr)
        render_result(probs)

    TIPS = '''
    <div style="
        background: rgba(255, 253, 248, 0.6);
        border-left: 4px solid #87A878;
        padding: 14px 20px;
        border-radius: 8px;
        font-size: 14px;
        line-height: 1.6;
        margin-top: 28px;
    ">
        <strong style="color: #5A7A56;">Tips for best results:</strong><br>
        • Try the RAVDESS sentences: <em>"Kids are talking by the door"</em><br>
        &nbsp;&nbsp;or <em>"Dogs are sitting by the door"</em><br>
        • Speak with clear, expressive emotion<br>
        • Record in a quiet space, close to the mic<br>
        • Aim for 2–4 seconds of speech
    </div>
    '''
    st.markdown(TIPS, unsafe_allow_html=True)


with tab_upload:
    st.markdown(
        '<h3 style="color: #5A7A56; font-size: 18px; margin-top: 8px;">Upload an audio file</h3>',
        unsafe_allow_html=True,
    )
    st.markdown(
        '<p style="color: #7A6F62; font-style: italic; font-size: 14px; margin-bottom: 16px;">'
        '.wav, .mp3, .m4a, .flac, .ogg — any common format works.</p>',
        unsafe_allow_html=True,
    )

    uploaded_file = st.file_uploader(
        " ",
        type=["wav", "mp3", "m4a", "flac", "ogg"],
        label_visibility="collapsed",
    )

    if uploaded_file is not None:
        audio_bytes = uploaded_file.read()
        st.audio(audio_bytes)
        with st.spinner("Listening..."):
            waveform, sr = load_audio_from_bytes(audio_bytes)
            probs = predict(waveform, sr)
        render_result(probs)


# ============================================================
# Footer
# ============================================================
FOOTER = '''
<div style="
    text-align: center;
    color: #9C8E7E;
    font-size: 12px;
    font-style: italic;
    padding: 32px 0 16px 0;
    margin-top: 32px;
    border-top: 1px solid rgba(135, 168, 120, 0.2);
">
    Wav2Vec2 fine-tuned on RAVDESS  ·  trained on 24 actors, 8 emotions
</div>
'''
st.markdown(FOOTER, unsafe_allow_html=True)

Overwriting app.py


In [ ]:
import subprocess
import time
import re
import threading
import queue

# --- Start Streamlit in the background ---
streamlit_proc = subprocess.Popen(
    ["streamlit", "run", "app.py",
     "--server.port", "8501",
     "--server.headless", "true",
     "--server.enableCORS", "false",
     "--server.enableXsrfProtection", "false",
     "--browser.gatherUsageStats", "false"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

print("Starting Streamlit server...")
time.sleep(8)
print("Streamlit running on localhost:8501")
print()

# --- Start cloudflared tunnel ---
print("Opening Cloudflare tunnel...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8501",
     "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True,
    bufsize=1,
)

# --- Capture the public URL from cloudflared output ---
url_pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")

def stream_output(proc, q):
    for line in proc.stdout:
        q.put(line)

q = queue.Queue()
t = threading.Thread(target=stream_output, args=(tunnel_proc, q), daemon=True)
t.start()

found_url = None
deadline = time.time() + 30
while time.time() < deadline and found_url is None:
    try:
        line = q.get(timeout=1)
        m = url_pattern.search(line)
        if m:
            found_url = m.group(0)
    except queue.Empty:
        pass

if found_url:
    print()
    print("=" * 60)
    print("  🍃  YOUR APP IS LIVE")
    print("=" * 60)
    print(f"  Open this URL in a new tab:")
    print(f"  {found_url}")
    print("=" * 60)
    print()
    print("(Tunnel runs until you stop this cell.)")
else:
    print("⚠️  Couldn\'t auto-detect the tunnel URL within 30s.")
    print("    Streaming cloudflared output below — look for a https://*.trycloudflare.com line:")

# Continue streaming any remaining output so cloudflared stays alive
try:
    while True:
        try:
            line = q.get(timeout=1)
            print(line, end="")
        except queue.Empty:
            if tunnel_proc.poll() is not None:
                print("\nTunnel process exited.")
                break
except KeyboardInterrupt:
    print("\nStopping tunnel...")
    tunnel_proc.terminate()
    streamlit_proc.terminate()

Starting Streamlit server...
Streamlit running on localhost:8501

Opening Cloudflare tunnel...

  🍃  YOUR APP IS LIVE
  Open this URL in a new tab:
  https://prefix-assigned-removable-fabrics.trycloudflare.com

(Tunnel runs until you stop this cell.)
2026-04-26T00:27:25Z INF +--------------------------------------------------------------------------------------------+
2026-04-26T00:27:25Z INF Cannot determine default configuration path. No file [config.yml config.yaml] in [~/.cloudflared ~/.cloudflare-warp ~/cloudflare-warp /etc/cloudflared /usr/local/etc/cloudflared]
2026-04-26T00:27:25Z INF Version 2026.3.0 (Checksum 4a9e50e6d6d798e90fcd01933151a90bf7edd99a0a55c28ad18f2e16263a5c30)
2026-04-26T00:27:25Z INF GOOS: linux, GOVersion: go1.24.13, GoArch: amd64
2026-04-26T00:27:25Z INF Settings: map[ha-connections:1 no-autoupdate:true protocol:quic url:http://localhost:8501]
2026-04-26T00:27:25Z INF Generated Connector ID: 2760e46e-cd36-4103-8488-9a177312e8f0
2026-04-26T00:27:25Z INF Initia